# 🤖 AI-Powered PDF Organizer - Experiment Notebook

This comprehensive notebook demonstrates the complete workflow of the AI-Powered PDF Organizer project. We'll walk through each step from PDF text extraction to intelligent clustering and organization.

## 📋 Project Overview

**Goal**: Automatically extract, embed, cluster, and organize PDF files by semantic similarity using advanced machine learning techniques.

**Use Cases**: 
- 📚 Academic research paper organization
- ⚖️ Legal document categorization  
- 💼 Business report clustering
- 🏥 Healthcare document management

**Workflow**:
1. **Extract** text from PDF documents
2. **Generate** semantic embeddings using Sentence-BERT
3. **Cluster** documents by similarity using K-Means
4. **Organize** files into themed folders
5. **Visualize** and evaluate results

Let's get started! 🚀

## 1. 🛠️ Environment Setup and Dependencies

First, let's install and import all the required libraries for our AI-powered PDF organizer.

In [ ]:
# Install required packages (run this cell if packages are not installed)
# !pip install PyMuPDF sentence-transformers scikit-learn pandas numpy matplotlib seaborn plotly tqdm

import sys
import os
from pathlib import Path

# Add project src directory to Python path
project_root = Path.cwd().parent
src_dir = project_root / "src"
sys.path.insert(0, str(src_dir))

print(f"Project root: {project_root}")
print(f"Source directory: {src_dir}")
print(f"Python path updated: {str(src_dir) in sys.path}")

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine learning and text processing
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer

# PDF processing
import fitz  # PyMuPDF

# Utilities
import json
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Our custom modules
from extraction.pdf_extractor import PDFExtractor
from embeddings.embedding_generator import EmbeddingGenerator
from clustering.document_clusterer import DocumentClusterer
from organization.file_organizer import FileOrganizer

# Set style for visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"📦 Available models in SentenceTransformers: {list(EmbeddingGenerator.AVAILABLE_MODELS.keys())}")
print(f"🎯 Available clustering algorithms: {list(DocumentClusterer.CLUSTERING_ALGORITHMS.keys())}")

## 2. 📄 PDF Text Extraction with PyMuPDF

Let's create some sample PDF content and extract text from PDF files using our PDFExtractor class.

In [ ]:
# Initialize PDF extractor
extractor = PDFExtractor(
    min_text_length=50,  # Minimum text length for valid pages
    max_file_size_mb=50  # Maximum file size to process
)

# Create a sample directory structure for demonstration
sample_data_dir = project_root / "data" / "sample_pdfs"
sample_data_dir.mkdir(parents=True, exist_ok=True)

print(f"📁 Sample data directory: {sample_data_dir}")
print(f"🔍 Extractor configuration:")
print(f"   - Min text length: {extractor.min_text_length}")
print(f"   - Max file size: {extractor.max_file_size_mb} MB")

# Function to create sample PDF content (for demonstration)
def create_sample_pdf_content():
    """Create sample content to simulate different types of documents."""
    sample_documents = {
        "machine_learning_paper.pdf": """
        Machine Learning in Modern Applications
        
        Abstract: This paper discusses the applications of machine learning algorithms 
        in various domains including natural language processing, computer vision, 
        and predictive analytics. We explore supervised learning techniques such as 
        random forests, support vector machines, and neural networks.
        
        Introduction: Machine learning has revolutionized data analysis and pattern 
        recognition. Deep learning models have achieved breakthrough results in 
        image classification, speech recognition, and language translation tasks.
        """,
        
        "legal_contract.pdf": """
        Software License Agreement
        
        This Software License Agreement (Agreement) is entered into between 
        the Licensor and Licensee. The terms and conditions govern the use 
        of the software product.
        
        Terms of Use: The licensee agrees to use the software only for 
        authorized purposes. Any redistribution or modification requires 
        written permission from the licensor.
        
        Liability: The licensor shall not be liable for any damages arising 
        from the use of this software.
        """,
        
        "financial_report.pdf": """
        Quarterly Financial Report - Q3 2024
        
        Revenue Analysis: Total revenue for Q3 2024 reached $2.5 million, 
        representing a 15% increase from the previous quarter. Key revenue 
        drivers include increased sales in the technology sector.
        
        Expenses: Operating expenses totaled $1.8 million, including 
        personnel costs, marketing expenses, and research and development 
        investments.
        
        Profit Margins: Net profit margin improved to 28%, demonstrating 
        strong operational efficiency and cost management.
        """,
        
        "healthcare_research.pdf": """
        Clinical Study: Effectiveness of New Treatment Protocol
        
        Methodology: This randomized controlled trial examined 200 patients 
        over a 6-month period to evaluate treatment efficacy. Participants 
        were divided into treatment and control groups.
        
        Results: The new treatment protocol showed significant improvement 
        in patient outcomes with 85% success rate compared to 60% in the 
        control group. Side effects were minimal and well-tolerated.
        
        Conclusions: The new treatment protocol demonstrates superior 
        efficacy and safety profile for the target condition.
        """,
        
        "ai_research_paper.pdf": """
        Advances in Artificial Intelligence and Deep Learning
        
        Abstract: Recent developments in artificial intelligence have led to 
        breakthrough achievements in various domains. This paper reviews the 
        latest advances in deep learning architectures, transformer models, 
        and their applications in natural language processing.
        
        Neural Networks: Convolutional neural networks (CNNs) have shown 
        remarkable performance in computer vision tasks. Recurrent neural 
        networks (RNNs) and Long Short-Term Memory (LSTM) networks excel 
        in sequential data processing.
        
        Future Directions: The integration of AI with edge computing and 
        federated learning presents new opportunities for distributed 
        machine learning systems.
        """
    }
    
    return sample_documents

sample_docs = create_sample_pdf_content()
print(f"\n📚 Created {len(sample_docs)} sample document contents")
for filename in sample_docs.keys():
    print(f"   - {filename}")
    
print("\n💡 Note: In a real scenario, you would have actual PDF files to process.")

In [ ]:
# Simulate PDF extraction results (since we don't have actual PDF files)
def simulate_pdf_extraction(sample_documents):
    """Simulate PDF extraction results for demonstration."""
    extraction_results = []
    
    for filename, content in sample_documents.items():
        # Clean the content
        cleaned_content = extractor._clean_text(content)
        
        result = {
            'file_path': str(sample_data_dir / filename),
            'filename': filename,
            'full_text': cleaned_content,
            'page_texts': [{'page_number': 1, 'text': cleaned_content}],
            'page_count': 1,
            'total_pages': 1,
            'word_count': len(cleaned_content.split()),
            'char_count': len(cleaned_content),
            'metadata': {
                'title': filename.replace('_', ' ').replace('.pdf', ''),
                'file_size_mb': len(content) / (1024 * 1024),
            },
            'success': True,
            'error': None
        }
        extraction_results.append(result)
    
    return extraction_results

# Get extraction results
extraction_results = simulate_pdf_extraction(sample_docs)

# Display extraction summary
print("📊 PDF Extraction Summary")
print("=" * 40)
print(f"Total PDFs processed: {len(extraction_results)}")
print(f"Successfully extracted: {sum(1 for r in extraction_results if r['success'])}")

# Create a summary DataFrame
extraction_df = pd.DataFrame([
    {
        'Filename': result['filename'],
        'Word Count': result['word_count'],
        'Character Count': result['char_count'],
        'Status': '✅ Success' if result['success'] else '❌ Failed'
    }
    for result in extraction_results
])

print("\n📋 Extraction Details:")
display(extraction_df)

# Show sample extracted text
print("\n📄 Sample Extracted Text (first 200 characters):")
for result in extraction_results[:2]:
    print(f"\n🔸 {result['filename']}:")
    print(f"   {result['full_text'][:200]}...")

## 3. 🧹 Text Preprocessing and Cleaning

Now let's examine and improve our text preprocessing pipeline to ensure clean, standardized text for embedding generation.

In [ ]:
# Analyze text characteristics before and after cleaning
def analyze_text_characteristics(texts, labels):
    """Analyze characteristics of text data."""
    analysis = []
    
    for text, label in zip(texts, labels):
        analysis.append({
            'Document': label,
            'Length': len(text),
            'Words': len(text.split()),
            'Sentences': text.count('.') + text.count('!') + text.count('?'),
            'Avg_Word_Length': np.mean([len(word) for word in text.split()]),
            'Unique_Words': len(set(text.lower().split()))
        })
    
    return pd.DataFrame(analysis)

# Extract texts and filenames
original_texts = [result['full_text'] for result in extraction_results]
filenames = [result['filename'] for result in extraction_results]

# Analyze original text characteristics
text_analysis = analyze_text_characteristics(original_texts, filenames)
print("📊 Text Analysis Summary:")
display(text_analysis)

# Visualize text statistics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📈 Text Characteristics Analysis', fontsize=16, fontweight='bold')

# Word count distribution
axes[0, 0].bar(range(len(text_analysis)), text_analysis['Words'])
axes[0, 0].set_title('Word Count per Document')
axes[0, 0].set_xlabel('Document Index')
axes[0, 0].set_ylabel('Word Count')
axes[0, 0].set_xticks(range(len(text_analysis)))
axes[0, 0].set_xticklabels([f'Doc {i+1}' for i in range(len(text_analysis))], rotation=45)

# Character length distribution
axes[0, 1].bar(range(len(text_analysis)), text_analysis['Length'], color='orange')
axes[0, 1].set_title('Character Count per Document')
axes[0, 1].set_xlabel('Document Index')
axes[0, 1].set_ylabel('Character Count')
axes[0, 1].set_xticks(range(len(text_analysis)))
axes[0, 1].set_xticklabels([f'Doc {i+1}' for i in range(len(text_analysis))], rotation=45)

# Average word length
axes[1, 0].bar(range(len(text_analysis)), text_analysis['Avg_Word_Length'], color='green')
axes[1, 0].set_title('Average Word Length per Document')
axes[1, 0].set_xlabel('Document Index')
axes[1, 0].set_ylabel('Avg Word Length')
axes[1, 0].set_xticks(range(len(text_analysis)))
axes[1, 0].set_xticklabels([f'Doc {i+1}' for i in range(len(text_analysis))], rotation=45)

# Unique words ratio
unique_ratio = text_analysis['Unique_Words'] / text_analysis['Words']
axes[1, 1].bar(range(len(text_analysis)), unique_ratio, color='red')
axes[1, 1].set_title('Vocabulary Diversity (Unique Words Ratio)')
axes[1, 1].set_xlabel('Document Index')
axes[1, 1].set_ylabel('Unique Words Ratio')
axes[1, 1].set_xticks(range(len(text_analysis)))
axes[1, 1].set_xticklabels([f'Doc {i+1}' for i in range(len(text_analysis))], rotation=45)

plt.tight_layout()
plt.show()

print(f"\n📈 Overall Statistics:")
print(f"   Average words per document: {text_analysis['Words'].mean():.1f}")
print(f"   Average characters per document: {text_analysis['Length'].mean():.1f}")
print(f"   Average vocabulary diversity: {unique_ratio.mean():.2f}")

## 4. 🧠 Generate Embeddings with Sentence-BERT

Now we'll use Sentence-BERT to generate semantic embeddings from our preprocessed text. We'll compare different models and analyze embedding characteristics.

In [ ]:
# Compare different embedding models
embedding_models = ['all-MiniLM-L6-v2', 'all-mpnet-base-v2']
model_results = {}

print("🧠 Testing different Sentence-BERT models:")
print("=" * 50)

for model_name in embedding_models:
    print(f"\n⏳ Testing model: {model_name}")
    
    try:
        # Initialize embedding generator
        embedding_generator = EmbeddingGenerator(
            model_name=model_name,
            cache_embeddings=False  # Disable caching for comparison
        )
        
        # Generate embeddings
        start_time = time.time()
        embeddings, document_names = embedding_generator.generate_document_embeddings(
            extraction_results,
            batch_size=16,
            show_progress=False
        )
        end_time = time.time()
        
        processing_time = end_time - start_time
        
        # Store results
        model_results[model_name] = {
            'embeddings': embeddings,
            'document_names': document_names,
            'processing_time': processing_time,
            'embedding_size': embeddings.shape[1] if len(embeddings) > 0 else 0,
            'model_info': embedding_generator.get_model_info()
        }
        
        print(f"   ✅ Success!")
        print(f"   📏 Embedding dimension: {embeddings.shape[1]}")
        print(f"   ⏱️ Processing time: {processing_time:.2f}s")
        
    except Exception as e:
        print(f"   ❌ Failed: {str(e)}")
        model_results[model_name] = {'error': str(e)}

# Create comparison table
comparison_data = []
for model_name, results in model_results.items():
    if 'error' not in results:
        comparison_data.append({
            'Model': model_name,
            'Embedding Size': results['embedding_size'],
            'Processing Time (s)': f"{results['processing_time']:.2f}",
            'Documents': len(results['document_names']),
            'Description': results['model_info'].get('description', 'N/A')
        })

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    print("\n📊 Model Comparison:")
    display(comparison_df)

In [ ]:
# Select the best performing model for further analysis
selected_model = 'all-MiniLM-L6-v2'  # Fast and efficient for our demo
if selected_model in model_results and 'error' not in model_results[selected_model]:
    embeddings = model_results[selected_model]['embeddings']
    document_names = model_results[selected_model]['document_names']
    
    print(f"🎯 Selected model: {selected_model}")
    print(f"📊 Embedding statistics:")
    print(f"   Shape: {embeddings.shape}")
    print(f"   Min value: {embeddings.min():.4f}")
    print(f"   Max value: {embeddings.max():.4f}")
    print(f"   Mean: {embeddings.mean():.4f}")
    print(f"   Std: {embeddings.std():.4f}")
    
    # Visualize embedding distributions
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Histogram of embedding values
    axes[0].hist(embeddings.flatten(), bins=50, alpha=0.7, edgecolor='black')
    axes[0].set_title('Distribution of Embedding Values')
    axes[0].set_xlabel('Embedding Value')
    axes[0].set_ylabel('Frequency')
    axes[0].grid(True, alpha=0.3)
    
    # Heatmap of embeddings (first 50 dimensions for visibility)
    n_dims_to_show = min(50, embeddings.shape[1])
    im = axes[1].imshow(embeddings[:, :n_dims_to_show], aspect='auto', cmap='viridis')
    axes[1].set_title(f'Embedding Heatmap (First {n_dims_to_show} Dimensions)')
    axes[1].set_xlabel('Embedding Dimension')
    axes[1].set_ylabel('Document Index')
    axes[1].set_yticks(range(len(document_names)))
    axes[1].set_yticklabels([name[:20] + '...' if len(name) > 20 else name for name in document_names])
    
    plt.colorbar(im, ax=axes[1])
    plt.tight_layout()
    plt.show()
    
    # Calculate similarity matrix
    embedding_generator = EmbeddingGenerator(model_name=selected_model)
    similarity_matrix = embedding_generator.compute_similarity_matrix(embeddings)
    
    print(f"\n🔗 Similarity Matrix:")
    print(f"   Shape: {similarity_matrix.shape}")
    print(f"   Diagonal (self-similarity): {np.diag(similarity_matrix)}")
    
    # Visualize similarity matrix
    plt.figure(figsize=(10, 8))
    mask = np.triu(np.ones_like(similarity_matrix, dtype=bool))
    sns.heatmap(similarity_matrix, 
                mask=mask,
                annot=True, 
                fmt='.3f',
                xticklabels=[name[:15] + '...' if len(name) > 15 else name for name in document_names],
                yticklabels=[name[:15] + '...' if len(name) > 15 else name for name in document_names],
                cmap='coolwarm',
                center=0.5,
                square=True)
    plt.title('📊 Document Similarity Matrix\n(Cosine Similarity between Embeddings)')
    plt.tight_layout()
    plt.show()
    
else:
    print(f"❌ Could not use model {selected_model}. Check previous results.")

## 5. 🎯 Clustering with K-Means

Now we'll apply clustering algorithms to group similar documents together based on their semantic embeddings.

In [ ]:
# Test different numbers of clusters to find optimal value
def evaluate_clustering_range(embeddings, k_range, algorithm='kmeans'):
    """Evaluate clustering performance for different k values."""
    results = []
    
    for k in k_range:
        if k >= len(embeddings):
            continue
            
        try:
            # Create clusterer
            clusterer = DocumentClusterer(
                algorithm=algorithm,
                n_clusters=k,
                random_state=42
            )
            
            # Fit and predict
            labels = clusterer.fit_predict(embeddings, auto_tune=False)
            
            # Calculate metrics
            if len(set(labels)) > 1:
                silhouette = silhouette_score(embeddings, labels)
                calinski_harabasz = calinski_harabasz_score(embeddings, labels)
            else:
                silhouette = 0
                calinski_harabasz = 0
            
            # Get inertia for K-Means
            inertia = clusterer.clusterer.inertia_ if hasattr(clusterer.clusterer, 'inertia_') else None
            
            results.append({
                'k': k,
                'silhouette_score': silhouette,
                'calinski_harabasz_score': calinski_harabasz,
                'inertia': inertia,
                'n_clusters_found': len(set(labels))
            })
            
        except Exception as e:
            print(f"Error with k={k}: {str(e)}")
    
    return pd.DataFrame(results)

# Evaluate different cluster counts
k_range = range(2, min(len(embeddings), 6))  # Test 2 to 5 clusters
clustering_evaluation = evaluate_clustering_range(embeddings, k_range)

print("🔍 Clustering Evaluation Results:")
display(clustering_evaluation)

# Visualize clustering metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Silhouette Score
axes[0].plot(clustering_evaluation['k'], clustering_evaluation['silhouette_score'], 
             marker='o', linewidth=2, markersize=8, color='blue')
axes[0].set_title('Silhouette Score vs Number of Clusters')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Silhouette Score')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(bottom=0)

# Calinski-Harabasz Score
axes[1].plot(clustering_evaluation['k'], clustering_evaluation['calinski_harabasz_score'], 
             marker='s', linewidth=2, markersize=8, color='green')
axes[1].set_title('Calinski-Harabasz Score vs Number of Clusters')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Calinski-Harabasz Score')
axes[1].grid(True, alpha=0.3)

# Inertia (Elbow Method)
if clustering_evaluation['inertia'].notna().any():
    axes[2].plot(clustering_evaluation['k'], clustering_evaluation['inertia'], 
                 marker='^', linewidth=2, markersize=8, color='red')
    axes[2].set_title('Inertia vs Number of Clusters (Elbow Method)')
    axes[2].set_xlabel('Number of Clusters (k)')
    axes[2].set_ylabel('Inertia')
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find optimal number of clusters
optimal_k_silhouette = clustering_evaluation.loc[clustering_evaluation['silhouette_score'].idxmax(), 'k']
optimal_k_calinski = clustering_evaluation.loc[clustering_evaluation['calinski_harabasz_score'].idxmax(), 'k']

print(f"\n🎯 Optimal Cluster Analysis:")
print(f"   Best k by Silhouette Score: {optimal_k_silhouette}")
print(f"   Best k by Calinski-Harabasz Score: {optimal_k_calinski}")

# Choose final k (prefer Silhouette score)
final_k = int(optimal_k_silhouette)
print(f"   🏆 Selected k: {final_k}")

In [ ]:
# Perform final clustering with optimal parameters
print(f"🎯 Performing final clustering with k={final_k}")

final_clusterer = DocumentClusterer(
    algorithm='kmeans',
    n_clusters=final_k,
    random_state=42
)

# Fit and predict
cluster_labels = final_clusterer.fit_predict(embeddings, auto_tune=False)

# Create detailed clustering results
clustering_df = final_clusterer.create_cluster_summary(document_names, embeddings)

print(f"\n📊 Final Clustering Results:")
print(f"   Number of clusters: {len(set(cluster_labels))}")
print(f"   Silhouette Score: {final_clusterer.silhouette_score:.3f}")
print(f"   Calinski-Harabasz Score: {final_clusterer.calinski_harabasz_score:.1f}")

# Display clustering results
print(f"\n📋 Cluster Assignments:")
display(clustering_df)

# Show cluster distribution
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print(f"\n📈 Cluster Distribution:")
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(cluster_labels)) * 100
    print(f"   Cluster {cluster_id}: {count} documents ({percentage:.1f}%)")

# Visualize cluster distribution
plt.figure(figsize=(10, 6))
bars = plt.bar(cluster_counts.index, cluster_counts.values, 
               color=plt.cm.Set3(np.linspace(0, 1, len(cluster_counts))))
plt.title('📊 Document Distribution Across Clusters')
plt.xlabel('Cluster ID')
plt.ylabel('Number of Documents')
plt.grid(True, alpha=0.3)

# Add value labels on bars
for bar, count in zip(bars, cluster_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
             str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Analyze cluster characteristics
print(f"\n🔍 Detailed Cluster Analysis:")
for cluster_id in sorted(set(cluster_labels)):
    cluster_docs = clustering_df[clustering_df['cluster_id'] == cluster_id]
    print(f"\n📁 Cluster {cluster_id}:")
    print(f"   Size: {len(cluster_docs)} documents")
    print(f"   Documents:")
    for doc in cluster_docs['document_name']:
        print(f"     - {doc}")
    
    # Show similarity within cluster
    if 'intra_cluster_similarity' in cluster_docs.columns:
        avg_similarity = cluster_docs['intra_cluster_similarity'].iloc[0]
        print(f"   Average intra-cluster similarity: {avg_similarity:.3f}")

## 6. 📊 Evaluate Clustering Performance

Let's thoroughly evaluate our clustering results and validate the quality of our document groupings.

In [ ]:
# Comprehensive clustering evaluation
def comprehensive_cluster_evaluation(embeddings, labels, document_names):
    """Perform comprehensive evaluation of clustering results."""
    
    evaluation_results = {}
    
    # Basic metrics
    n_clusters = len(set(labels))
    n_documents = len(labels)
    
    # Clustering quality metrics
    if n_clusters > 1:
        silhouette_avg = silhouette_score(embeddings, labels)
        calinski_harabasz = calinski_harabasz_score(embeddings, labels)
    else:
        silhouette_avg = 0
        calinski_harabasz = 0
    
    # Per-sample silhouette scores
    if n_clusters > 1:
        silhouette_samples = silhouette_score(embeddings, labels, metric='cosine')
    else:
        silhouette_samples = [0] * n_documents
    
    # Intra-cluster distances
    intra_cluster_distances = []
    inter_cluster_distances = []
    
    for cluster_id in set(labels):
        cluster_mask = labels == cluster_id
        cluster_embeddings = embeddings[cluster_mask]
        
        if len(cluster_embeddings) > 1:
            # Pairwise distances within cluster
            from sklearn.metrics.pairwise import cosine_distances
            intra_distances = cosine_distances(cluster_embeddings)
            # Get upper triangle (excluding diagonal)
            n = len(intra_distances)
            if n > 1:
                upper_indices = np.triu_indices(n, k=1)
                intra_cluster_distances.extend(intra_distances[upper_indices])
    
    evaluation_results = {
        'n_clusters': n_clusters,
        'n_documents': n_documents,
        'silhouette_score': silhouette_avg,
        'calinski_harabasz_score': calinski_harabasz,
        'avg_intra_cluster_distance': np.mean(intra_cluster_distances) if intra_cluster_distances else 0,
        'std_intra_cluster_distance': np.std(intra_cluster_distances) if intra_cluster_distances else 0
    }
    
    return evaluation_results

# Perform comprehensive evaluation
evaluation = comprehensive_cluster_evaluation(embeddings, cluster_labels, document_names)

print("🔬 Comprehensive Clustering Evaluation")
print("=" * 45)
for metric, value in evaluation.items():
    if isinstance(value, float):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

# Create evaluation metrics visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('🔍 Clustering Performance Metrics', fontsize=16, fontweight='bold')

# Silhouette analysis
if evaluation['n_clusters'] > 1:
    from sklearn.metrics import silhouette_samples
    silhouette_vals = silhouette_samples(embeddings, cluster_labels)
    
    y_lower = 10
    for i, cluster_id in enumerate(sorted(set(cluster_labels))):
        cluster_silhouette_vals = silhouette_vals[cluster_labels == cluster_id]
        cluster_silhouette_vals.sort()
        
        size_cluster_i = cluster_silhouette_vals.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = plt.cm.tab10(i)
        axes[0, 0].fill_betweenx(np.arange(y_lower, y_upper),
                                0, cluster_silhouette_vals,
                                facecolor=color, edgecolor=color, alpha=0.7)
        
        axes[0, 0].text(-0.05, y_lower + 0.5 * size_cluster_i, str(cluster_id))
        y_lower = y_upper + 10
    
    axes[0, 0].axvline(x=evaluation['silhouette_score'], color="red", linestyle="--", 
                       label=f'Average Score: {evaluation["silhouette_score"]:.3f}')
    axes[0, 0].set_xlabel('Silhouette Coefficient Values')
    axes[0, 0].set_ylabel('Cluster Label')
    axes[0, 0].set_title('Silhouette Analysis')
    axes[0, 0].legend()

# Cluster size distribution
cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index()
axes[0, 1].pie(cluster_sizes.values, labels=[f'Cluster {i}' for i in cluster_sizes.index], 
               autopct='%1.1f%%', startangle=90)
axes[0, 1].set_title('Cluster Size Distribution')

# Distance from cluster centers
if hasattr(final_clusterer.clusterer, 'cluster_centers_'):
    centers = final_clusterer.clusterer.cluster_centers_
    distances_to_centers = []
    
    for i, embedding in enumerate(embeddings):
        cluster_id = cluster_labels[i]
        center = centers[cluster_id]
        distance = np.linalg.norm(embedding - center)
        distances_to_centers.append(distance)
    
    axes[1, 0].hist(distances_to_centers, bins=20, alpha=0.7, edgecolor='black')
    axes[1, 0].axvline(np.mean(distances_to_centers), color='red', linestyle='--',
                       label=f'Mean: {np.mean(distances_to_centers):.3f}')
    axes[1, 0].set_xlabel('Distance to Cluster Center')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distribution of Distances to Cluster Centers')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

# Cluster compactness comparison
cluster_compactness = []
for cluster_id in sorted(set(cluster_labels)):
    cluster_mask = cluster_labels == cluster_id
    cluster_embeddings = embeddings[cluster_mask]
    
    if len(cluster_embeddings) > 1:
        # Calculate average pairwise distance within cluster
        from sklearn.metrics.pairwise import euclidean_distances
        distances = euclidean_distances(cluster_embeddings)
        upper_triangle = np.triu_indices(len(distances), k=1)
        avg_distance = np.mean(distances[upper_triangle])
    else:
        avg_distance = 0
    
    cluster_compactness.append(avg_distance)

axes[1, 1].bar(range(len(cluster_compactness)), cluster_compactness, 
               color=plt.cm.Set3(np.linspace(0, 1, len(cluster_compactness))))
axes[1, 1].set_xlabel('Cluster ID')
axes[1, 1].set_ylabel('Average Intra-Cluster Distance')
axes[1, 1].set_title('Cluster Compactness')
axes[1, 1].set_xticks(range(len(cluster_compactness)))
axes[1, 1].set_xticklabels([f'C{i}' for i in sorted(set(cluster_labels))])
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Quality assessment
print(f"\n📈 Quality Assessment:")
if evaluation['silhouette_score'] > 0.5:
    quality = "🟢 Excellent"
elif evaluation['silhouette_score'] > 0.3:
    quality = "🟡 Good"
elif evaluation['silhouette_score'] > 0.1:
    quality = "🟠 Fair" 
else:
    quality = "🔴 Poor"

print(f"   Overall clustering quality: {quality}")
print(f"   Silhouette score interpretation:")
print(f"     > 0.5: Strong clustering structure")
print(f"     0.3-0.5: Reasonable clustering structure") 
print(f"     0.1-0.3: Weak clustering structure")
print(f"     < 0.1: No meaningful clustering structure")

## 7. 🗂️ Organize PDFs by Clusters

Now let's organize our PDF files based on the clustering results and create a structured file organization system.

In [ ]:
# Setup file organization
output_dir = project_root / "data" / "organized_output"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"🗂️ Setting up file organization")
print(f"Output directory: {output_dir}")

# Initialize file organizer
organizer = FileOrganizer(
    base_output_dir=str(output_dir),
    create_timestamp_dir=True,
    copy_files=True  # Copy files instead of moving them
)

print(f"Organizer output directory: {organizer.output_dir}")

# Generate descriptive cluster names based on content analysis
def generate_cluster_themes(clustering_df, extraction_results):
    """Generate thematic names for clusters based on document content."""
    cluster_themes = {}
    
    # Create mapping from filename to full text
    text_mapping = {result['filename']: result['full_text'] for result in extraction_results}
    
    for cluster_id in sorted(clustering_df['cluster_id'].unique()):
        cluster_docs = clustering_df[clustering_df['cluster_id'] == cluster_id]
        
        # Analyze common words in cluster documents
        all_words = []
        for doc_name in cluster_docs['document_name']:
            if doc_name in text_mapping:
                text = text_mapping[doc_name].lower()
                # Extract meaningful words (skip common words)
                words = [word for word in text.split() 
                        if len(word) > 4 and word not in ['that', 'this', 'with', 'from', 'they', 'have', 'been']]
                all_words.extend(words)
        
        # Count word frequencies
        word_counts = {}
        for word in all_words:
            word_counts[word] = word_counts.get(word, 0) + 1
        
        # Get top keywords
        top_keywords = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:3]
        
        if top_keywords and top_keywords[0][1] > 1:  # At least appears twice
            theme_words = [word for word, count in top_keywords if count > 1]
            if 'machine' in theme_words or 'learning' in theme_words or 'artificial' in theme_words:
                theme = "AI_Machine_Learning"
            elif 'legal' in theme_words or 'contract' in theme_words or 'agreement' in theme_words:
                theme = "Legal_Documents"
            elif 'financial' in theme_words or 'revenue' in theme_words or 'profit' in theme_words:
                theme = "Financial_Reports"
            elif 'clinical' in theme_words or 'treatment' in theme_words or 'patient' in theme_words:
                theme = "Healthcare_Research"
            else:
                # Use most common meaningful word
                theme = theme_words[0].title() if theme_words else f"Category_{cluster_id}"
        else:
            theme = f"Mixed_Documents_{cluster_id}"
        
        cluster_themes[cluster_id] = theme
    
    return cluster_themes

# Generate cluster themes
cluster_themes = generate_cluster_themes(clustering_df, extraction_results)

print(f"\n🏷️ Generated Cluster Themes:")
for cluster_id, theme in cluster_themes.items():
    cluster_size = len(clustering_df[clustering_df['cluster_id'] == cluster_id])
    print(f"   Cluster {cluster_id}: {theme} ({cluster_size} documents)")

# Simulate organization (since we don't have actual files to move)
print(f"\n📁 Organization Plan:")
print("=" * 50)

organization_plan = {}
for cluster_id in sorted(clustering_df['cluster_id'].unique()):
    cluster_docs = clustering_df[clustering_df['cluster_id'] == cluster_id]
    theme = cluster_themes[cluster_id]
    
    folder_name = f"cluster_{cluster_id}_{theme}"
    organization_plan[cluster_id] = {
        'folder_name': folder_name,
        'theme': theme,
        'documents': cluster_docs['document_name'].tolist(),
        'count': len(cluster_docs)
    }
    
    print(f"\n📂 {folder_name}/")
    for doc in cluster_docs['document_name']:
        print(f"   📄 {doc}")

# Create organization summary report
organization_summary = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'total_documents': len(clustering_df),
    'total_clusters': len(set(clustering_df['cluster_id'])),
    'clustering_quality': {
        'silhouette_score': float(final_clusterer.silhouette_score),
        'calinski_harabasz_score': float(final_clusterer.calinski_harabasz_score)
    },
    'cluster_details': organization_plan
}

# Save organization summary
summary_file = organizer.output_dir / "reports" / "organization_plan.json"
with open(summary_file, 'w') as f:
    json.dump(organization_summary, f, indent=2)

print(f"\n💾 Organization summary saved to: {summary_file}")

# Create visual organization chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Cluster size distribution with themes
cluster_data = [(cluster_id, info['count'], info['theme']) 
                for cluster_id, info in organization_plan.items()]
cluster_ids, counts, themes = zip(*cluster_data)

colors = plt.cm.Set3(np.linspace(0, 1, len(cluster_ids)))
bars = ax1.bar(cluster_ids, counts, color=colors)
ax1.set_title('📊 Documents per Organized Folder')
ax1.set_xlabel('Cluster ID')
ax1.set_ylabel('Number of Documents')
ax1.grid(True, alpha=0.3)

# Add value labels and theme info
for i, (bar, count, theme) in enumerate(zip(bars, counts, themes)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             str(count), ha='center', va='bottom', fontweight='bold')
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
             theme.replace('_', '\n'), ha='center', va='center', 
             fontsize=8, rotation=0, fontweight='bold', color='white')

# Pie chart of document distribution
ax2.pie(counts, labels=[f'C{cid}\n{theme}' for cid, theme in zip(cluster_ids, themes)], 
        autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title('📈 Document Distribution by Theme')

plt.tight_layout()
plt.show()

print(f"\n✅ File organization planning completed!")
print(f"📋 Summary:")
print(f"   - Total documents: {len(clustering_df)}")
print(f"   - Organized into: {len(organization_plan)} themed folders")
print(f"   - Average documents per folder: {len(clustering_df)/len(organization_plan):.1f}")
print(f"   - Organization quality (Silhouette): {final_clusterer.silhouette_score:.3f}")

## 8. 📊 Visualize Clustering Results

Let's create comprehensive visualizations to understand and present our clustering results using dimensionality reduction techniques.

In [ ]:
# Create comprehensive visualizations of clustering results

# 1. PCA Visualization
print("📊 Creating PCA visualization...")
pca = PCA(n_components=2, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)

# 2. t-SNE Visualization (if we have enough samples)
print("📊 Creating t-SNE visualization...")
if len(embeddings) >= 4:  # t-SNE requires at least 4 samples
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)-1))
    embeddings_tsne = tsne.fit_transform(embeddings)
else:
    print("   ⚠️ Not enough samples for t-SNE, using PCA instead")
    embeddings_tsne = embeddings_pca

# Create interactive Plotly visualizations
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('PCA Projection', 't-SNE Projection', 
                   'Cluster Similarity Heatmap', 'Cluster Statistics'),
    specs=[[{"type": "scatter"}, {"type": "scatter"}],
           [{"type": "heatmap"}, {"type": "bar"}]]
)

# Prepare data
cluster_colors = px.colors.qualitative.Set3[:len(set(cluster_labels))]
color_map = {cluster_id: color for cluster_id, color in zip(sorted(set(cluster_labels)), cluster_colors)}

# PCA Plot
for cluster_id in sorted(set(cluster_labels)):
    mask = cluster_labels == cluster_id
    cluster_theme = cluster_themes.get(cluster_id, f'Cluster {cluster_id}')
    
    fig.add_trace(
        go.Scatter(
            x=embeddings_pca[mask, 0],
            y=embeddings_pca[mask, 1],
            mode='markers+text',
            marker=dict(
                color=color_map[cluster_id],
                size=12,
                line=dict(width=2, color='black')
            ),
            text=[name[:10] + '...' if len(name) > 10 else name for name in np.array(document_names)[mask]],
            textposition="top center",
            name=f'{cluster_theme}',
            hovertemplate='<b>%{text}</b><br>PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra></extra>'
        ),
        row=1, col=1
    )

# t-SNE Plot
for cluster_id in sorted(set(cluster_labels)):
    mask = cluster_labels == cluster_id
    cluster_theme = cluster_themes.get(cluster_id, f'Cluster {cluster_id}')
    
    fig.add_trace(
        go.Scatter(
            x=embeddings_tsne[mask, 0],
            y=embeddings_tsne[mask, 1],
            mode='markers+text',
            marker=dict(
                color=color_map[cluster_id],
                size=12,
                line=dict(width=2, color='black')
            ),
            text=[name[:10] + '...' if len(name) > 10 else name for name in np.array(document_names)[mask]],
            textposition="top center",
            name=f'{cluster_theme}',
            showlegend=False,
            hovertemplate='<b>%{text}</b><br>t-SNE1: %{x:.3f}<br>t-SNE2: %{y:.3f}<extra></extra>'
        ),
        row=1, col=2
    )

# Similarity Heatmap
similarity_matrix = np.corrcoef(embeddings)
fig.add_trace(
    go.Heatmap(
        z=similarity_matrix,
        x=[name[:15] + '...' if len(name) > 15 else name for name in document_names],
        y=[name[:15] + '...' if len(name) > 15 else name for name in document_names],
        colorscale='RdYlBu_r',
        showscale=True,
        hovertemplate='Doc1: %{x}<br>Doc2: %{y}<br>Similarity: %{z:.3f}<extra></extra>'
    ),
    row=2, col=1
)

# Cluster Statistics
cluster_stats = []
for cluster_id in sorted(set(cluster_labels)):
    cluster_size = np.sum(cluster_labels == cluster_id)
    cluster_stats.append({
        'Cluster': f'C{cluster_id}: {cluster_themes.get(cluster_id, "")}',
        'Size': cluster_size,
        'Percentage': (cluster_size / len(cluster_labels)) * 100
    })

cluster_stats_df = pd.DataFrame(cluster_stats)

fig.add_trace(
    go.Bar(
        x=cluster_stats_df['Cluster'],
        y=cluster_stats_df['Size'],
        marker_color=[color_map[i] for i in sorted(set(cluster_labels))],
        text=[f'{size}<br>({pct:.1f}%)' for size, pct in zip(cluster_stats_df['Size'], cluster_stats_df['Percentage'])],
        textposition='inside',
        hovertemplate='%{x}<br>Documents: %{y}<br>Percentage: %{text}<extra></extra>'
    ),
    row=2, col=2
)

# Update layout
fig.update_layout(
    title_text="🤖 AI-Powered PDF Organizer - Clustering Results Dashboard",
    title_x=0.5,
    height=800,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Update axes labels
fig.update_xaxes(title_text=f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)", row=1, col=1)
fig.update_yaxes(title_text=f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)", row=1, col=1)
fig.update_xaxes(title_text="t-SNE Dimension 1", row=1, col=2)
fig.update_yaxes(title_text="t-SNE Dimension 2", row=1, col=2)
fig.update_xaxes(title_text="Cluster", row=2, col=2)
fig.update_yaxes(title_text="Number of Documents", row=2, col=2)

fig.show()

# Additional static matplotlib visualizations
fig_static, axes = plt.subplots(2, 2, figsize=(16, 12))
fig_static.suptitle('📊 Detailed Clustering Analysis', fontsize=16, fontweight='bold')

# 1. PCA with cluster centers and confidence ellipses
ax = axes[0, 0]
for cluster_id in sorted(set(cluster_labels)):
    mask = cluster_labels == cluster_id
    cluster_points = embeddings_pca[mask]
    
    # Plot points
    ax.scatter(cluster_points[:, 0], cluster_points[:, 1], 
              label=f'Cluster {cluster_id}', alpha=0.7, s=100)
    
    # Plot cluster center
    center = np.mean(cluster_points, axis=0)
    ax.scatter(center[0], center[1], marker='x', s=200, 
              color='black', linewidth=3)
    
    # Add confidence ellipse
    if len(cluster_points) > 1:
        from matplotlib.patches import Ellipse
        cov = np.cov(cluster_points.T)
        eigenvals, eigenvecs = np.linalg.eig(cov)
        angle = np.degrees(np.arctan2(eigenvecs[1, 0], eigenvecs[0, 0]))
        
        ellipse = Ellipse(center, 2*np.sqrt(eigenvals[0]), 2*np.sqrt(eigenvals[1]),
                         angle=angle, alpha=0.3)
        ax.add_patch(ellipse)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA with Confidence Ellipses')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Cluster quality metrics comparison
ax = axes[0, 1]
quality_metrics = ['Silhouette Score', 'Calinski-Harabasz Score (scaled)']
quality_values = [
    final_clusterer.silhouette_score,
    final_clusterer.calinski_harabasz_score / 100  # Scale for comparison
]

bars = ax.bar(quality_metrics, quality_values, color=['skyblue', 'lightcoral'])
ax.set_title('Clustering Quality Metrics')
ax.set_ylabel('Score')
ax.grid(True, alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars, quality_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
           f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

# 3. Document similarity network (simplified)
ax = axes[1, 0]
similarity_threshold = 0.3  # Show connections above this similarity

# Create adjacency matrix
adj_matrix = similarity_matrix > similarity_threshold
np.fill_diagonal(adj_matrix, False)  # Remove self-connections

# Position documents in a circle
angles = np.linspace(0, 2*np.pi, len(document_names), endpoint=False)
positions = [(np.cos(angle), np.sin(angle)) for angle in angles]

# Draw connections
for i in range(len(document_names)):
    for j in range(i+1, len(document_names)):
        if adj_matrix[i, j]:
            x1, y1 = positions[i]
            x2, y2 = positions[j]
            ax.plot([x1, x2], [y1, y2], 'gray', alpha=0.5, linewidth=1)

# Draw nodes
for i, (x, y) in enumerate(positions):
    cluster_id = cluster_labels[i]
    color = plt.cm.Set3(cluster_id / len(set(cluster_labels)))
    ax.scatter(x, y, c=[color], s=200, edgecolors='black', linewidth=2)
    ax.annotate(document_names[i][:8] + '...', (x, y), 
               xytext=(5, 5), textcoords='offset points', fontsize=8)

ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_aspect('equal')
ax.set_title(f'Document Similarity Network (threshold: {similarity_threshold})')
ax.axis('off')

# 4. Explained variance and feature importance
ax = axes[1, 1]
cumulative_var = np.cumsum(pca.explained_variance_ratio_[:min(10, len(pca.explained_variance_ratio_))])
components = range(1, len(cumulative_var) + 1)

ax.plot(components, cumulative_var, 'bo-', linewidth=2, markersize=8)
ax.axhline(y=0.95, color='r', linestyle='--', alpha=0.7, label='95% Variance')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('PCA Explained Variance')
ax.grid(True, alpha=0.3)
ax.legend()

# Add annotations
for i, var in enumerate(cumulative_var):
    ax.annotate(f'{var:.2f}', (components[i], var), 
               xytext=(0, 10), textcoords='offset points', 
               ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# Print final summary
print("🎉 Clustering Analysis Complete!")
print("=" * 50)
print(f"📊 Results Summary:")
print(f"   • Total documents processed: {len(document_names)}")
print(f"   • Clusters identified: {len(set(cluster_labels))}")
print(f"   • Clustering quality (Silhouette): {final_clusterer.silhouette_score:.3f}")
print(f"   • PCA explained variance: {sum(pca.explained_variance_ratio_[:2]):.1%}")

print(f"\n🗂️ Organization Summary:")
for cluster_id, info in organization_plan.items():
    print(f"   • {info['folder_name']}: {info['count']} documents")

print(f"\n💡 Insights:")
print(f"   • Documents successfully grouped by semantic similarity")
print(f"   • {len(set(cluster_labels))} distinct themes identified")
print(f"   • Ready for automated file organization")

## 🎉 Conclusion and Next Steps

Congratulations! You've successfully built and experimented with an AI-powered PDF organizer. Here's what we accomplished:

### ✅ What We Built
1. **📄 PDF Text Extraction**: Robust text extraction with error handling
2. **🧠 Semantic Embeddings**: Advanced NLP using Sentence-BERT models  
3. **🎯 Intelligent Clustering**: Automated grouping with optimal parameter detection
4. **📊 Performance Evaluation**: Comprehensive quality metrics and validation
5. **🗂️ Smart Organization**: Automated file organization with thematic naming
6. **📈 Rich Visualizations**: Interactive and static plots for analysis

### 🏆 Key Results
- Successfully clustered documents by semantic similarity
- Achieved clustering quality score of **{:.3f}** (Silhouette Score)
- Identified **{}** distinct document themes
- Created organized folder structure for efficient file management

### 🚀 Potential Enhancements
- **🔧 Advanced Features**: Add support for more file types (Word, PowerPoint, etc.)
- **🌐 Web Deployment**: Deploy the Streamlit app for broader accessibility  
- **📱 Mobile Support**: Create mobile-friendly interface
- **🤖 ML Improvements**: Experiment with more advanced models (BERT, RoBERTa)
- **⚡ Performance**: Implement batch processing for large document collections
- **🔍 Search**: Add semantic search capabilities within organized documents

### 📚 Learning Resources
- [Sentence-Transformers Documentation](https://www.sbert.net/)
- [Scikit-learn Clustering Guide](https://scikit-learn.org/stable/modules/clustering.html)
- [Streamlit Documentation](https://docs.streamlit.io/)

Ready to organize your PDF collection with AI! 🤖📚